# MVP — Análise de Dados e Boas Práticas

**Movimentação de óleo diesel A S10 em terminais aquaviários do Sudeste, 2025**

MBA PUC-Rio · Versão de trabalho · Validação inicial da base

# 1. Problema e perguntas

O objetivo é descrever os fluxos de recepção e entrega de diesel A S10 informados pelos terminais do Sudeste em 2025. A unidade de observação é um registro mensal de volume para uma combinação de terminal, produto, sentido, tipo de operação e modo de transporte. A unicidade dessa combinação será verificada na análise exploratória.

- **H1.** Como o volume mensal de recepção aquaviária de diesel A S10 se distribuiu entre os terminais do Sudeste em 2025?
- **H2.** Qual foi a participação de cada modo de transporte nas entregas de diesel A S10 registradas, por terminal com entregas reportadas?

A análise é descritiva e mantém recepções e entregas separadas. A base reserva é a série mensal de vendas de derivados e etanol da ANP.

# 2. Fonte e significado dos dados

[Movimentação dos Terminais Aquaviários — ANP](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/movimentacao-dos-terminais-aquaviarios) · [Dicionário oficial](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-movimentacao-dos-terminais-aquaviarios/metadados-movimentacao-terminais-aquaviarios.pdf)

A carga usa o arquivo público da ANP. A fonte pode ser atualizada. O código registra o SHA-256 para identificar os bytes usados em cada execução. A política de redistribuição e o procedimento de fixação da versão serão verificados antes da entrega.

| Variável | Significado | Tipo / unidade |
|---|---|---|
| mes_de_referencia | Mês de referência | Data mensal |
| codigo_anp_do_terminal | Identificador do estabelecimento na ANP | Código |
| uf | UF do terminal | Categoria |
| sentido_da_operacao | Recepção (1) ou entrega (2) | Código |
| tipo_da_operacao | Categoria da operação, conforme dicionário | Código |
| modo_de_transporte | Rodoviário (1), ferroviário (2), aquaviário (4), dutoviário (5) ou outros (9) | Código |
| codigo_anp_do_produto | Identificador do produto | Código |
| descricao_do_produto | Descrição informada do produto | Texto |
| volume_m3 | Volume informado a 20 °C | m³ |

# 3. Ambiente e carregamento

Execute as células em ordem com Python, pandas e openpyxl. A fonte é baixada sem autenticação. O formato é identificado pelos bytes do arquivo, pois a extensão publicada pode divergir do conteúdo.

In [ ]:
from hashlib import sha256
from importlib.metadata import version
from io import BytesIO
import sys
from urllib.request import urlopen

import pandas as pd

URL_ANP = (
    "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/"
    "arquivos/arquivos-movimentacao-dos-terminais-aquaviarios/"
    "dados-abertos-movimentacao-terminais-aquaviarios.csv"
)

print("Python", sys.version.split()[0])
for pacote in ("pandas", "openpyxl"):
    print(pacote, version(pacote))

with urlopen(URL_ANP, timeout=60) as resposta:
    arquivo = resposta.read()

print("SHA-256", sha256(arquivo).hexdigest())

if arquivo.startswith(b"PK\x03\x04"):
    dados_origem = pd.read_excel(BytesIO(arquivo), engine="openpyxl")
    print("Formato recebido: Excel")
else:
    dados_origem = pd.read_csv(
        BytesIO(arquivo), sep=";", decimal=",", encoding="utf-8-sig"
    )
    print("Formato recebido: CSV")

dados = dados_origem.rename(
    columns=lambda nome: nome.strip().lower()
).rename(columns={"mesreferencia": "mes_de_referencia"})

colunas_necessarias = {
    "mes_de_referencia", "codigo_anp_do_terminal", "uf",
    "sentido_da_operacao", "tipo_da_operacao", "modo_de_transporte",
    "codigo_anp_do_produto", "descricao_do_produto", "volume_m3",
}
faltantes = colunas_necessarias.difference(dados.columns)
if faltantes:
    raise ValueError(f"Colunas esperadas ausentes: {sorted(faltantes)}")

mes = pd.to_datetime(dados["mes_de_referencia"], format="mixed", errors="coerce")
if (mes.isna() & dados["mes_de_referencia"].notna()).any():
    raise ValueError("Há meses de referência em formato inesperado.")

sudeste_2025 = dados.loc[
    mes.dt.year.eq(2025) & dados["uf"].isin(["ES", "MG", "RJ", "SP"])
].copy()
sudeste_2025["mes_de_referencia"] = mes.loc[sudeste_2025.index].dt.to_period("M")
if sudeste_2025["mes_de_referencia"].nunique() != 12:
    raise ValueError("O recorte regional não contém os 12 meses de 2025.")

codigo_produto = pd.to_numeric(sudeste_2025["codigo_anp_do_produto"], errors="coerce")
diesel_s10 = sudeste_2025.loc[codigo_produto.eq(420105001)].copy()
if diesel_s10.empty:
    raise ValueError("O produto 420105001 não foi encontrado no recorte.")

# 4. Análise exploratória

Checagem inicial de cobertura e descrições do produto.

In [ ]:
resumo_carga = pd.DataFrame({
    "Registros": [len(dados_origem), len(sudeste_2025), len(diesel_s10)],
    "Terminais": [
        dados["codigo_anp_do_terminal"].nunique(),
        sudeste_2025["codigo_anp_do_terminal"].nunique(),
        diesel_s10["codigo_anp_do_terminal"].nunique(),
    ],
}, index=["Arquivo de origem", "Sudeste em 2025", "Diesel A S10 no recorte"])

display(resumo_carga)
display(diesel_s10["descricao_do_produto"].value_counts(dropna=False))

## 4.1 Granularidade, ausências e cobertura

A chave candidata serve para investigar repetições. A tabela de cobertura conta registros por terminal e mês. Zero na tabela indica ausência de registros no recorte.

In [ ]:
chave_candidata = [
    "mes_de_referencia", "codigo_anp_do_terminal", "codigo_anp_do_produto",
    "sentido_da_operacao", "tipo_da_operacao", "modo_de_transporte",
]

ausencias = diesel_s10.isna().sum().rename("ausências")
display(pd.concat([diesel_s10.dtypes.rename("tipo"), ausencias], axis=1))

print("Duplicatas exatas além da primeira ocorrência:", diesel_s10.duplicated().sum())
repeticoes = diesel_s10.duplicated(subset=chave_candidata, keep=False)
print("Registros em chaves candidatas repetidas:", repeticoes.sum())
if repeticoes.any():
    display(
        diesel_s10.loc[repeticoes]
        .groupby(chave_candidata, dropna=False)
        .agg(registros=("volume_m3", "size"), volumes_distintos=("volume_m3", "nunique"))
        .sort_values("registros", ascending=False)
        .head(10)
    )

meses_2025 = pd.period_range("2025-01", "2025-12", freq="M")
cobertura = (
    diesel_s10.groupby(["codigo_anp_do_terminal", "mes_de_referencia"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=meses_2025, fill_value=0)
)
display(cobertura.assign(meses_com_registro=cobertura.gt(0).sum(axis=1)))

# 5. Pré-processamento e decisões

A primeira comparação prevista é entre seleção por descrição literal e por código ANP, usando contagem de registros e volume em m³. As outras decisões serão fundamentadas após o diagnóstico. A análise é descritiva e não prevê divisão em treino e teste.

# 6. Respostas e conclusão

Seção a ser elaborada pelo aluno após a análise, com evidências e limites para cada pergunta.

# 7. Checklist de entrega

A revisão final inclui execução em sessão nova no Colab, verificação da versão da fonte, checklist com localizações, referências, declaração de uso de IA e conferência dos links de entrega.